# Ejercicio 3 — Regresion Lineal Multiple con NumPy
## Practica Final: Estadística para Data Science

**Autor:** Alejandro Pujana Quintero  
**Proposito:** Implementar OLS desde cero con NumPy y validar contra sklearn.

---

### Por que este ejercicio es el mas dificil

Los ejercicios 1, 2 y 4 usan herramientas existentes (sklearn, statsmodels, seasonal_decompose).  
Este ejercicio pide **construir la herramienta desde cero** usando solo algebra matricial con NumPy.

Hay tres capas de dificultad:
1. **Matematica matricial**: traducir la formula beta = (X'X)^-1 * X'y a operaciones NumPy
2. **Columna de unos**: agregar el intercepto manualmente — sklearn lo hace automatico con `fit_intercept=True`
3. **Estabilidad numerica**: saber usar `lstsq` en lugar de `inv` para evitar matrices singulares

### Datos sinteticos — no el dataset Loan Approval

Este ejercicio usa datos **completamente sinteticos** generados con semilla=42.  
Los coeficientes reales son conocidos de antemano:

| Parametro | Valor real | Descripcion |
|---|---|---|
| β₀ | 5.0 | Intercepto |
| β₁ | 2.0 | Coeficiente feature 1 |
| β₂ | -1.0 | Coeficiente feature 2 |
| β₃ | 0.5 | Coeficiente feature 3 |
| σ (ruido) | 1.5 | Desviacion tipica del ruido gaussiano |

El modelo generador es: `y = 5.0 + 2.0*x1 - 1.0*x2 + 0.5*x3 + N(0, 1.5)`  
El objetivo es recuperar estos coeficientes a partir de los datos observados.

## Derivacion matematica de la formula OLS

### Por que minimizar la suma de cuadrados

El modelo lineal es: `y = X*beta + epsilon`  
donde epsilon es el error (ruido). No podemos observar epsilon, pero podemos minimizar
los residuos `e = y - X*beta_estimado`.

OLS minimiza la **Suma de Residuos al Cuadrado**:

```
SRC(beta) = e'e = (y - X*beta)' * (y - X*beta)
```

### Derivacion paso a paso

**Paso 1** — Expandir el producto:
```
SRC(beta) = y'y - 2*y'*X*beta + beta'*X'*X*beta
```

**Paso 2** — Condicion de primer orden (derivar e igualar a cero):
```
d(SRC)/d(beta) = -2*X'*y + 2*X'*X*beta = 0
```

**Paso 3** — Ecuaciones normales:
```
X'*X * beta = X'*y
```

**Paso 4** — Solucion analitica (si X'X es invertible):
```
beta = (X'X)^-1 * X'y
```

### Por que np.linalg.lstsq en lugar de np.linalg.inv

`inv()` falla si X'X es singular (multicolinealidad perfecta) o casi singular (mal condicionada).  
`lstsq()` resuelve el sistema X'X * beta = X'y usando **descomposicion SVD** — siempre converge  
y da la misma solucion cuando X'X es invertible. Es la opcion numericamente robusta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Importar las funciones implementadas desde el script de entrega
sys.path.insert(0, str(Path('..') / 'practica_final_Pujana_Quintero_Alejandro'))
from ejercicio3_regresion_multiple import (
    regresion_lineal_multiple, calcular_mae, calcular_rmse, calcular_r2
)

# Generar los mismos datos sinteticos que el main del profesor
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

n_muestras  = 200
n_features  = 3
X           = rng.standard_normal((n_muestras, n_features))
coefs_reales = np.array([5.0, 2.0, -1.0, 0.5])
ruido       = rng.normal(0, 1.5, n_muestras)
y           = coefs_reales[0] + X @ coefs_reales[1:] + ruido

# Split 80/20 sin shuffle — exactamente como el main del profesor
corte = int(0.8 * n_muestras)
X_train, X_test = X[:corte], X[corte:]
y_train, y_test = y[:corte], y[corte:]

print(f'Dataset sintetico generado con semilla={SEMILLA}')
print(f'Total: {n_muestras} obs | Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Coeficientes reales conocidos: {coefs_reales}')
print(f'Ruido gaussiano: N(0, 1.5)')

## TODO 1 — regresion_lineal_multiple

### Que hace cada paso de la implementacion

**Paso 1 — Columna de unos:**
```python
ones_train = np.ones((n_train, 1))
X_train_b  = np.hstack([ones_train, X_train])
```
Sin esta columna, el modelo asume beta0=0 y los coeficientes quedan sesgados.  
La matriz de diseno X_train_b tiene forma (160, 4): columna de 1s + 3 features.

**Paso 2 — lstsq resuelve las ecuaciones normales:**
```python
coefs, _, _, _ = np.linalg.lstsq(X_train_b, y_train, rcond=None)
```
lstsq minimiza `||X_train_b @ coefs - y_train||^2` usando SVD.  
Devuelve 4 valores: (coeficientes, residuos, rango, valores singulares).  
Solo usamos el primero — los coeficientes beta.

**Paso 3 — Predicciones:**
```python
X_test_b = np.hstack([np.ones((n_test, 1)), X_test])
y_pred   = X_test_b @ coefs
```
Producto matricial: cada fila de X_test_b multiplicada por el vector de coeficientes.

In [ ]:
# Ejecutar la implementacion NumPy
coefs, y_pred = regresion_lineal_multiple(X_train, y_train, X_test)

print('Comparativa de coeficientes:')
print(f'{"Parametro":<20} {"Real":>10} {"Ajustado":>12} {"Diferencia":>12}')
print('-' * 56)
nombres = ['b0 (intercepto)', 'b1 (feature 1)', 'b2 (feature 2)', 'b3 (feature 3)']
for nombre, real, ajust in zip(nombres, coefs_reales, coefs):
    diff = ajust - real
    print(f'{nombre:<20} {real:>10.4f} {ajust:>12.6f} {diff:>+12.6f}')

### Interpretacion de los coeficientes ajustados

| Parametro | Real | Ajustado | Diferencia | Interpretacion |
|---|---|---|---|---|
| β₀ | 5.000 | 4.8650 | -0.135 | Intercepto: valor de y cuando todas las x=0 |
| β₁ | 2.000 | 2.0636 | +0.064 | Por cada unidad de x1, y aumenta ~2 unidades |
| β₂ | -1.000 | -1.1170 | -0.117 | Por cada unidad de x2, y disminuye ~1 unidad |
| β₃ | 0.500 | 0.4385 | -0.062 | Por cada unidad de x3, y aumenta ~0.5 unidades |

**Las diferencias son pequenas y esperables.** Con solo 160 observaciones de entrenamiento
y ruido gaussiano sigma=1.5, los estimadores OLS no recuperan los valores exactos — tienen
varianza de muestreo. La diferencia maxima es de 0.135 en beta0, dentro del margen esperado.

**Los signos son correctos**: beta1>0 (positivo), beta2<0 (negativo), beta3>0 (positivo).  
El modelo recupera correctamente la direccion del efecto de cada variable.

## TODO 2 — Metricas sin sklearn

### Implementacion desde la formula matematica

**MAE** = mean(|y_real - y_pred|)  
Error promedio en las mismas unidades del target. Robusto a outliers.
```python
return np.mean(np.abs(y_real - y_pred))
```

**RMSE** = sqrt(mean((y_real - y_pred)²))  
Penaliza errores grandes mas que el MAE. Si RMSE >> MAE hay predicciones muy malas.
```python
return np.sqrt(np.mean((y_real - y_pred) ** 2))
```

**R²** = 1 - SS_res / SS_tot  
Proporcion de varianza explicada. R²=1 perfecto, R²=0 sin mejor que la media, R²<0 peor.
```python
ss_res = np.sum((y_real - y_pred) ** 2)
ss_tot = np.sum((y_real - np.mean(y_real)) ** 2)
return 1 - ss_res / ss_tot
```

In [ ]:
mae  = calcular_mae(y_test, y_pred)
rmse = calcular_rmse(y_test, y_pred)
r2   = calcular_r2(y_test, y_pred)

print('Metricas NumPy OLS vs referencia del profesor:')
print(f'{"Metrica":<8} {"Obtenido":>10} {"Referencia":>14} {"Dentro del rango":>18}')
print('-' * 54)
print(f'{"MAE":<8} {mae:>10.4f} {"~1.20 (+-0.20)":>14} {"Si" if 1.0 <= mae <= 1.4 else "No":>18}')
print(f'{"RMSE":<8} {rmse:>10.4f} {"~1.50 (+-0.20)":>14} {"Si" if 1.3 <= rmse <= 1.7 else "No":>18}')
print(f'{"R2":<8} {r2:>10.4f} {"~0.80 (+-0.05)":>14} {"Si" if 0.75 <= r2 <= 0.85 else "Nota: ver abajo":>18}')

print(f'\nNota sobre R2={r2:.4f} vs referencia ~0.80:')
print('El test set tiene solo 40 observaciones (20% de 200).')
print('Con muestras pequenas, la varianza del R2 es alta.')
print('La VALIDACION OK con sklearn confirma que la implementacion es correcta.')

### Interpretacion de las metricas

**MAE = 1.1665**: el modelo se equivoca en promedio 1.17 unidades al predecir y.  
Dentro del rango de referencia (1.20 ±0.20). Correcto.

**RMSE = 1.4612**: ligeramente mayor que el MAE, indicando algunos errores grandes pero sin casos extremos. Dentro del rango (1.50 ±0.20). Correcto.

**R² = 0.6897**: el modelo explica el 68.97% de la varianza del target.  
Esto esta fuera del rango de referencia (~0.80 ±0.05) pero **no es un error de implementacion**.  
La causa: el test set tiene solo 40 observaciones. Con muestras pequenas, el R² tiene  
alta varianza muestral y puede diferir del valor esperado.  
La prueba definitiva es la validacion contra sklearn: diferencia = 0.00000000 en todas las metricas.

**Relacion con el ruido teorico**: con sigma=1.5 y la magnitud de los coeficientes,  
un R² de ~0.80 es el valor esperado con n=200. Con solo 40 observaciones en test,  
la estimacion del R² tiene mas incertidumbre. Si usaramos el train set (160 obs),
el R² seria mas cercano al valor de referencia.

## Validacion comparativa — NumPy OLS vs sklearn

### Por que hacemos esta validacion

El profesor indico que los resultados de NumPy deben ser casi identicos a sklearn  
ya que ambos implementan exactamente el mismo algoritmo OLS.  
Si los coeficientes difieren, hay un error en la implementacion NumPy.

Esta validacion es la prueba de correctitud mas rigurosa que podemos hacer.

In [ ]:
from sklearn.linear_model import LinearRegression

modelo_sk = LinearRegression()
modelo_sk.fit(X_train, y_train)
y_pred_sk = modelo_sk.predict(X_test)
coefs_sk  = np.concatenate([[modelo_sk.intercept_], modelo_sk.coef_])

mae_sk  = np.mean(np.abs(y_test - y_pred_sk))
rmse_sk = np.sqrt(np.mean((y_test - y_pred_sk)**2))
r2_sk   = 1 - np.sum((y_test-y_pred_sk)**2) / np.sum((y_test-np.mean(y_test))**2)

print(f'{"Metrica":<8} {"NumPy OLS":>12} {"sklearn":>12} {"Diferencia":>14}')
print('-' * 50)
print(f'{"MAE":<8} {mae:>12.6f} {mae_sk:>12.6f} {abs(mae-mae_sk):>14.8f}')
print(f'{"RMSE":<8} {rmse:>12.6f} {rmse_sk:>12.6f} {abs(rmse-rmse_sk):>14.8f}')
print(f'{"R2":<8} {r2:>12.6f} {r2_sk:>12.6f} {abs(r2-r2_sk):>14.8f}')
print(f'\nCoeficientes:')
print(f'{"":20} {"NumPy":>12} {"sklearn":>12} {"Diferencia":>14}')
for nombre, cn, cs in zip(nombres, coefs, coefs_sk):
    print(f'  {nombre:<20} {cn:>12.6f} {cs:>12.6f} {abs(cn-cs):>14.8f}')

if np.allclose(coefs, coefs_sk, atol=1e-6):
    print('\nVALIDACION OK: implementacion NumPy identica a sklearn (tolerancia=1e-6)')
    print('Ambos algoritmos resuelven exactamente el mismo sistema OLS.')

### Interpretacion de la validacion

**Diferencia = 0.00000000 en todos los coeficientes y metricas.**

Esto confirma que la implementacion NumPy es matematicamente identica a sklearn.  
Ambos resuelven el mismo sistema de ecuaciones normales X'X * beta = X'y.

sklearn internamente usa `np.linalg.lstsq` o una descomposicion SVD equivalente —  
la misma estrategia que implementamos. La diferencia de 0.0 no es coincidencia:  
son el mismo algoritmo ejecutado sobre los mismos datos.

## TODO 3 — Grafico Real vs Predicho

### Como interpretar este grafico

El eje X muestra los valores reales de y (conocidos, del test set).  
El eje Y muestra las predicciones del modelo para esas mismas observaciones.

La **linea roja discontinua** es la prediccion perfecta: y_pred = y_real.  
Si el modelo fuera perfecto, todos los puntos estarian exactamente sobre esta linea.

La **distancia vertical** de cada punto a la linea = residuo de esa observacion.  
Puntos por encima de la linea: el modelo sobreestima (predice mas de lo real).  
Puntos por debajo de la linea: el modelo subestima (predice menos de lo real).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_test, y_pred, alpha=0.65, s=40, color='#1976D2',
           edgecolors='white', linewidth=0.4, label='Predicciones NumPy OLS')

lim_min = min(y_test.min(), y_pred.min()) - 1
lim_max = max(y_test.max(), y_pred.max()) + 1
ax.plot([lim_min, lim_max], [lim_min, lim_max],
        color='crimson', linewidth=1.5, linestyle='--', label='Prediccion perfecta (y=x)')

ax.set_xlim(lim_min, lim_max)
ax.set_ylim(lim_min, lim_max)
ax.set_xlabel('Valores Reales')
ax.set_ylabel('Valores Predichos')
ax.set_title(
    f'Real vs Predicho — OLS NumPy\n'
    f'MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}',
    fontweight='bold'
)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Estadisticos de los residuos
residuos = y_test - y_pred
print(f'Estadisticos de los residuos:')
print(f'  Media  : {residuos.mean():.6f}  (ideal: 0)')
print(f'  Std    : {residuos.std():.4f}')
print(f'  Min/Max: {residuos.min():.4f} / {residuos.max():.4f}')

### Interpretacion del grafico

**Patron general**: los puntos siguen la diagonal, confirmando que el modelo captura
la relacion lineal correctamente. No hay curvatura ni patron sistematico.

**Dispersion**: la nube de puntos tiene una dispersion aproximadamente uniforme
alrededor de la diagonal — consistente con el ruido gaussiano homoscedastico
con el que se generaron los datos (sigma=1.5).

**Contraste con ejercicio 2**: el grafico de residuos del ejercicio 2 mostraba
heterocedasticidad (patron de abanico). Este grafico no muestra ese patron porque:
1. Los datos son sinteticos con ruido gaussiano puro — no hay heterocedasticidad estructural
2. No hay variable dominante como `loan_to_income_ratio` que genere predicciones en bandas

**Algunos puntos alejados**: hay observaciones con residuos grandes (ej. el punto
en x~-1, y~0.4). Esto es normal con solo 40 observaciones de test — el azar
puede generar algunos casos con ruido mayor al tipico.

## Respuestas para Respuestas.md

### Pregunta 3.1 — Que hace beta = (X'X)^-1 * X'y y por que la columna de unos

La formula beta = (X'X)^-1 * X'y es la solucion analitica que minimiza la suma de
cuadrados de los residuos. Deriva de la condicion de primer orden del problema de
minimizacion: d(SRC)/d(beta) = 0, que da las ecuaciones normales X'X * beta = X'y.
Despejando beta se obtiene la formula cerrada.

La columna de unos es necesaria porque permite que el modelo tenga un intercepto beta0
(termino independiente). Sin ella, la recta de regresion pasa forzosamente por el origen
(0,0), sesgando todos los coeficientes. Al agregar una columna de 1s, el producto
X_b @ beta = beta0*1 + beta1*x1 + ... incluye el termino constante automaticamente.

### Pregunta 3.2 — Coeficientes ajustados vs referencia

| Parametro | Valor real | Valor ajustado | Diferencia |
|---|---|---|---|
| β₀ | 5.0 | 4.864995 | -0.135 |
| β₁ | 2.0 | 2.063618 | +0.064 |
| β₂ | -1.0 | -1.117038 | -0.117 |
| β₃ | 0.5 | 0.438517 | -0.062 |

Los coeficientes son proximos a los reales. Las diferencias son consecuencia del ruido
gaussiano con sigma=1.5 en los datos — con mas observaciones, convergirian a los valores reales.

### Pregunta 3.3 — Metricas y comparativa con referencia

MAE=1.1665 (ref: ~1.20), RMSE=1.4612 (ref: ~1.50), R²=0.6897 (ref: ~0.80).  
MAE y RMSE dentro del rango de referencia. R² ligeramente inferior — atribuible a la
varianza muestral con solo 40 observaciones en test. La validacion contra sklearn
confirma correctitud: diferencia = 0.00000000 en todos los coeficientes y metricas.

### Pregunta 3.4 — Comparativa con regresion del ejercicio 2

El ejercicio 2 usaba el dataset Loan Approval (50,000 obs, 22 features, datos reales).
Este ejercicio usa datos sinteticos (200 obs, 3 features, relacion lineal perfecta + ruido).

Diferencias clave:
- R² ejercicio 2: 0.817 (datos reales, mas features). R² ejercicio 3: 0.69 (test set pequeno)
- Residuos ejercicio 2: heterocedasticos (patron de abanico visible). Ejercicio 3: homocedasticos
- Ejercicio 2: solo 6/22 variables significativas. Ejercicio 3: las 3 features son relevantes por diseno
- La validacion NumPy=sklearn con diferencia=0 confirma que ambos algoritmos son identicos matematicamente